In [0]:
docs_table = "idp.processed.docs_chunked"
# Enable Change Data Feed for vector search synchronization
spark.sql(f"ALTER TABLE {docs_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
# Display sample data to understand table structure
display(spark.sql (f"SELECT * FROM {docs_table} LIMIT 5"))

path,chunk,id
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"== page == # Riverbend Marketing Studio 22 Kingfisher Ave Nashville, TN 37203 (615) 555-0172 • invoices@riverbendstudio.com Website: riverbendstudio.com ## INVOICE",0
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,- **Invoice #:** INV-20250010 - **Invoice Date:** 2025-01-17 - **Due Date:** 2025-01-17 - **Terms:** Due on Receipt - **PO #:** PO-13352 - **Currency:** USD,1
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,- **Payment Method:** Wire Transfer,2
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"| Bill To | Ship To | |---------|---------| | | | **Keystone Logistics Partners** 3400 River Rd Harrisburg, PA 17110",3
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,| Description | Qty | Unit Price | Amount | |-------------|-----|------------|--------| | Creative design (hours) | 8 | $93.48 | $747.84 | | Ad spend management (hours) | 12 | $78.47 | $941.64 |,4


In [0]:
# Compute embeddings 
import mlflow.deployments

# Initialize deployment client for accessing embedding models
deploy_client = mlflow.deployments.get_deploy_client("databricks")
# Generate embeddings for a sample question
question = "How Generative AI impacts humans?"
response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [question]})
embeddings = [e["embedding"] for e in response.data]
# Display embedding information
print ("Embedding for question:", embeddings [0]) 
print ("Embedding shape:", len (embeddings [0]))

Embedding for question: [1.201171875, -0.02044677734375, -0.4375, 0.156494140625, -0.250732421875, -0.50927734375, 0.96044921875, -1.4052734375, -1.638671875, 0.1719970703125, -0.2384033203125, -0.1661376953125, 0.69921875, -0.2437744140625, -0.6474609375, -0.35986328125, 0.88134765625, -0.69189453125, 0.485107421875, -0.9775390625, -0.12396240234375, -1.1875, 0.4208984375, -0.701171875, 1.2626953125, -0.371826171875, -0.2374267578125, -0.59619140625, 0.496337890625, -0.1798095703125, 1.1171875, 0.423095703125, -0.72509765625, 0.51953125, 0.281982421875, 0.19091796875, 0.7890625, -0.235107421875, -0.41259765625, 0.78857421875, 0.5380859375, 0.058746337890625, -0.2169189453125, -0.49853515625, -0.487548828125, -0.254150390625, 0.51123046875, -1.076171875, -0.57666015625, 0.3505859375, -0.2412109375, -0.6044921875, 0.48046875, -0.2122802734375, -0.41455078125, -0.428466796875, 0.6171875, -0.499267578125, -0.5263671875, 0.3330078125, -0.164794921875, -0.157958984375, 0.370361328125, -0.15

In [0]:
%pip install -U databricks-vectorsearch databricks-sdk
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.0/814.0 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-16c111d0-3ed5-4639-b4c4-dfe7b486723d
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.49.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-16c111d0-3ed5-4639-b4c4-dfe7b486723d
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-api-core 2.20.0 requires protobuf!=3.20.0,!=3.20.1,!

In [0]:
# Create Vector index using SDK (In Databricks on AWS project I used UI method to create Vector Search Index)
from databricks.vector_search.client import VectorSearchClient

# Initialize the Vector Search client
vsc = VectorSearchClient (disable_notice=True)

# Define index name using three-level naming convention
index_name = "idp.processed.docs_chunked_vs_index"
vector_search_endpoint = "vs_endpoint_test"
docs_table = "idp.processed.docs_chunked"

# Create the index using managed embeddings with Delta Sync
vsc.create_delta_sync_index_and_wait(
    endpoint_name=vector_search_endpoint, 
    index_name=index_name, 
    source_table_name=docs_table, 
    primary_key='id', 
    embedding_source_column="chunk", 
    embedding_model_endpoint_name="databricks-gte-large-en", 
    pipeline_type="TRIGGERED",
)

print (f"Index '{index_name}' created for table '{docs_table}' using endpoint '{vector_search_endpoint}'.")

Index 'idp.processed.docs_chunked_vs_index' created for table 'idp.processed.docs_chunked' using endpoint 'vs_endpoint_test'.


In [0]:
# Get the vector search index for performing searches
index = vsc.get_index(index_name=index_name)
print(index.describe())

{'name': 'idp.processed.docs_chunked_vs_index', 'endpoint_name': 'vs_endpoint_test', 'primary_key': 'id', 'index_type': 'DELTA_SYNC', 'delta_sync_index_spec': {'source_table': 'idp.processed.docs_chunked', 'embedding_source_columns': [{'name': 'chunk', 'embedding_model_endpoint_name': 'databricks-gte-large-en'}], 'pipeline_type': 'TRIGGERED', 'pipeline_id': '64e4c512-45a9-46a9-a0b1-2a0d4c3c8201'}, 'status': {'detailed_state': 'ONLINE_NO_PENDING_UPDATE', 'message': 'Index creation succeeded. Check latest status: https://dbc-e9ebba49-5f13.cloud.databricks.com/explore/data/idp/processed/docs_chunked_vs_index', 'indexed_row_count': 150, 'triggered_update_status': {'last_processed_commit_version': 1, 'last_processed_commit_timestamp': '2026-03-08T01:46:18Z'}, 'ready': True, 'index_url': 'dbc-e9ebba49-5f13.cloud.databricks.com/api/2.0/vector-search/indexes/idp.processed.docs_chunked_vs_index'}, 'creator': 'pallavinishanth@gmail.com', 'endpoint_type': 'STANDARD', 'id': '14aad57f-21da-4149-a18

In [0]:
# Search type: Similarity search
query_text = "any one bought docking station?"
results = index.similarity_search(
    query_text=query_text,
    columns= ["path", "chunk"], 
    num_results=3
)
display(results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 3,
  'data_array': [['dbfs:/Volumes/idp/default/final_project/purchase_order_pinecrest_25.pdf',
    '| Line | Description | Qty | Unit Price | Amount |\n|------|-------------|-----|------------|--------|\n| 1 | Laptop docking station | 3 | $1,138.86 | $3,416.58 |',
    0.6411285669696748],
   ['dbfs:/Volumes/idp/default/final_project/purchase_order_summit_25.pdf',
    '| Line | Description | Qty | Unit Price | Amount |\n|------|-------------|-----|------------|--------|\n| 1 | Laptop docking station | 1 | $1,372.25 | $1,372.25 |',
    0.6358685129677788],
   ['dbfs:/Volumes/idp/default/final_project/purchase_order_keystone_25.pdf',
    '| 2 | Standing desk converter | 3 | $1,364.02 | $4,092.06 |\n| 3 | Laptop docking station | 3 | $287.67 | $863.01 |\n| 4 | Replacement parts kit | 1 | $188.39 | $188.39 |',
    0.6234323685156838]]}}

In [0]:
# Search type: Hybrid search
query_text = "Get invoice with invoice number INV-20250007"
results_hybrid = index.similarity_search(
    query_text=query_text,
    columns= ["path", "chunk"],
    query_type="hybrid",
    num_results=5
)
display(results_hybrid)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 5,
  'data_array': [['dbfs:/Volumes/idp/default/final_project/invoice_7266442.pdf',
    '- **Invoice #:** INV‑20250007  \n- **Invoice Date:** 2025‑01‑22  \n- **Due Date:** 2025‑01‑22  \n- **Terms:** Due on Receipt  \n- **PO #:** PO‑16532  \n- **Currency:** USD',
    1.0],
   ['dbfs:/Volumes/idp/default/final_project/invoice_16234.pdf',
    '- **Invoice #:** INV-20250001  \n- **Invoice Date:** 2025-01-01  \n- **Due Date:** 2025-01-01  \n- **Terms:** Due on Receipt  \n- **PO #:** PO-17238  \n- **Currency:** USD',
    0.9077380952380952],
   ['dbfs:/Volumes/idp/default/final_project/invoice_54674553.pdf',
    '- **Invoice #:** INV-20250005  \n- **Invoice Date:** 2025-01-07  \n- **Due Date:** 2025-01-07  \n- **Terms:** Due on Receipt  \n- **PO #:** PO-13728  \n- **Currency:** USD',
    0.9066666666666667],
   ['dbfs:/Volumes/idp/default/final_project/invoice_101

In [0]:
# search type: FULL_TEXT is not enabled yet.

In [0]:
# Re-ranker is also not enabled yet.
# Example: Re-rank top N results from a semantic search using DatabricksReranker
# from databricks. vector_search. reranker import DatabricksReranker
# query_text = "any one bought docking station?"
# results_reranked = index.similarity_search(
#     query_text=query_text,
#     columns=["path", "chunk"], num_results=5,
#     reranker=DatabricksReranker(columns_to_rerank=[" chunk" ])
# )
# display(results_reranked)